<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Bioinformatics_DCA/blob/master/13_clase_Ordenar_archivos_de_secuencias_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ordenar un archivo de secuencias (FASTA o FASTQ)

**Nota para Google Colab:**  
- Ejecuta `!pip install biopython` si aún no lo tienes instalado.  
- Se requiere conexión a internet para descargar el archivo de ejemplo.

Aquí queremos ordenar un archivo de secuencias por **longitud de secuencia**.

- Si el archivo es lo suficientemente pequeño, podemos cargarlo todo en memoria como una lista de objetos `SeqRecord`, ordenar la lista y guardarla.
- Cuando tenemos un archivo muy grande que no cabe en memoria, podemos usar `Bio.SeqIO.index()`, que presentamos en la clase anterior.

Vamos a mostrar cómo ordenar archivos de secuencias en ambos casos.

### Obtener los datos

Genoma de orquídea (archivo de ejemplo de Biopython).

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/biopython/biopython/master/Doc/examples/ls_orchid.fasta"
urllib.request.urlretrieve(url, "ls_orchid.fasta")
print("Archivo ls_orchid.fasta descargado exitosamente.")

### Trabajar con archivos pequeños

In [ ]:
!pip install biopython

In [ ]:
from Bio import SeqIO

records = list(SeqIO.parse("ls_orchid.fasta", "fasta"))
for record in records:
    print(len(record.seq))

In [ ]:
# Ordenar por longitud (ascendente).
# Para orden descendente (de la más larga a la más corta): key=lambda r: -len(r)
records.sort(key=lambda r: len(r))
SeqIO.write(records, "sorted_orchids.fasta", "fasta")

In [ ]:
for record in records:
    print(len(record.seq))

### Trabajar con archivos grandes

Cuando el archivo no cabe en memoria, no cargamos todos los registros a la vez.  
En su lugar:
1. Recorremos el archivo solo para obtener pares `(longitud, id)`.
2. Ordenamos esos pares.
3. Indexamos el archivo con `SeqIO.index()`.
4. Escribimos las secuencias en el orden deseado accediendo por ID.

In [ ]:
# Obtener longitudes e IDs, y ordenar por longitud
len_and_ids = sorted(
    (len(rec), rec.id) for rec in SeqIO.parse("ls_orchid.fasta", "fasta")
)
len_and_ids

In [ ]:
# Invertir para orden descendente (de la secuencia más larga a la más corta)
ids = reversed([id for (length, id) in len_and_ids])
del len_and_ids  # liberar esta memoria

In [ ]:
type(ids)

In [ ]:
print(list(ids))

In [ ]:
# Indexar el archivo FASTA
record_index = SeqIO.index("ls_orchid.fasta", "fasta")

In [ ]:
# Obtener las secuencias del diccionario indexado según el orden de IDs
records = (record_index[id] for id in ids)
SeqIO.write(records, "sorted.fasta", "fasta")
print("Archivo 'sorted.fasta' escrito exitosamente.")